# SocialPulse - Advanced Analysis (Week 5)

Time-series trends (volume, sentiment, topics), emerging-topic detection, top influencers, and model validation - on the full ~14.3k-post feature dataset. Trend tables are windowed to the recent dense period (`src/trends.py`); model metrics come from `data/reports/model_eval`.

Sentiment uses the production scorer (VADER); engagement is normalized within platform.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
CLEAN = Path("../data/clean")
TRENDS = Path("../data/reports/trends")
EVAL = Path("../data/reports/model_eval")
FIG = Path("../data/reports/eda/figures"); FIG.mkdir(parents=True, exist_ok=True)

## 1. Volume over time

In [ ]:
vol = pd.read_csv(TRENDS / "volume_over_time.csv")
piv = vol.pivot_table(index="week", columns="Platform", values="posts", aggfunc="sum").fillna(0)
ax = piv.plot(marker="o")
ax.set_title("Posts per week by platform"); ax.set_ylabel("posts"); ax.tick_params(axis="x", rotation=45)
plt.tight_layout(); plt.savefig(FIG / "trend_volume.png", dpi=120); plt.show()

## 2. Sentiment over time
Share of positive vs negative posts per week (English only).

In [ ]:
st = pd.read_csv(TRENDS / "sentiment_over_time.csv")
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, lab, color in [(axes[0], "positive", "#55a868"), (axes[1], "negative", "#c44e52")]:
    sub = st[st["sentiment_label"] == lab]
    for p, g in sub.groupby("Platform"):
        ax.plot(g["week"], g["share_pct"], marker="o", label=p)
    ax.set_title(f"{lab.title()} share (%) per week"); ax.tick_params(axis="x", rotation=45); ax.legend()
plt.tight_layout(); plt.savefig(FIG / "trend_sentiment.png", dpi=120); plt.show()

## 3. Emerging topics
Change in each topic's share, recent half vs earlier half of the window.

In [ ]:
em = pd.read_csv(TRENDS / "emerging_topics.csv").sort_values("change_pct_points")
colors = ["#c44e52" if v < 0 else "#55a868" for v in em["change_pct_points"]]
ax = em.set_index("dominant_topic_label")["change_pct_points"].plot(kind="barh", color=colors)
ax.set_title("Emerging (green) vs declining (red) topics"); ax.set_xlabel("change in share (pct points)")
plt.tight_layout(); plt.savefig(FIG / "trend_emerging_topics.png", dpi=120); plt.show()

## 4. Top influencers (by total engagement)

In [ ]:
df = pd.read_csv(CLEAN / "features_dataset.csv")
clean = df[df["Platform"].isin(["youtube", "instagram"])]
top = (clean.groupby(["Platform", "Author"])["engagement_raw"].sum()
       .sort_values(ascending=False).head(12).reset_index())
top

## 5. Model validation metrics
In-domain sentiment leaderboard, deploy-domain gold result, and topic coherence.

In [ ]:
print("In-domain sentiment (Twitter held-out):")
print(pd.read_csv(EVAL / "sentiment_benchmark.csv")[["model", "macro_f1", "accuracy"]].to_string(index=False))
print("\nDeploy-domain (YouTube/Instagram gold):")
print(pd.read_csv(EVAL / "sentiment_gold_eval.csv").to_string(index=False))
print("\nTopic model:")
print(pd.read_csv(EVAL / "topic_model_eval.csv").to_string(index=False))

## Key takeaways

- Volume is rising week over week as the daily collection accumulates.
- Instagram sentiment is far more positive than YouTube; both are positive-dominant.
- Business/startup is the largest and fastest-rising topic; a ChatGPT-usage topic is also rising.
- Engagement is essentially independent of sentiment (validated at full scale).
- Validation: VADER is the production sentiment scorer (best deploy-domain macro-F1); NMF beats LDA on coherence.
- See `data/reports/insights_summary.md` for the auto-generated factual rollup; interpretation and recommendations go in the final report.